# Speculative decoding report — Qwen3-4B

Analysis of one sweep run written by `python -m experiments.run_sweep` (see `experiments/README.md`
for the record schemas). The notebook is run-agnostic: set `RUN_DIR` in the next cell and *Run All*.

**Sections**
1. Load the run and check data integrity
2. Metric definitions and aggregation
3. Performance comparison — acceptance, decode latency, speedup, latency breakdown, step-level explanation
4. Startup costs (prefill / drafter initialization), kept out of the decode comparison
5. Resource efficiency — memory, VRAM scaling, latency–memory trade-offs, work per generated token
6. Notes on methodology and design decisions

In [ ]:
# ---- Configuration: the only cell that needs editing --------------------------------------------
RUN_DIR = None          # e.g. "experiments/results/20261025T162558Z_qwen3_4b_smoke"; None = newest run
RESULTS_ROOT = "experiments/results"   # relative paths are resolved against the repository root
N_BOOT = 2000           # bootstrap replicates (resampling samples) for confidence intervals
CI_LEVEL = 0.95
SEED = 0                # bootstrap seed; with a fixed seed every number in the report is reproducible
SAVE_DIR = None         # e.g. "experiments/results/<run_id>/report" to also write CSV tables and PNG figures

In [ ]:
import json
import sys
import warnings
import zlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "experiments").is_dir() and (path / "deepspec").is_dir():
            return path
    raise FileNotFoundError(f"no DeepSpec repository above {start}")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from experiments.metrics import check_parity, read_jsonl  # noqa: E402


def resolve(path) -> Path:
    path = Path(path).expanduser()
    return path if path.is_absolute() else REPO_ROOT / path


if RUN_DIR is None:
    candidates = sorted(p for p in resolve(RESULTS_ROOT).iterdir() if (p / "manifest.json").is_file())
    if not candidates:
        raise FileNotFoundError(f"no runs under {resolve(RESULTS_ROOT)}")
    run_dir = candidates[-1]  # run ids start with a UTC timestamp
else:
    run_dir = resolve(RUN_DIR)
save_dir = None if SAVE_DIR is None else resolve(SAVE_DIR)
if save_dir is not None:
    save_dir.mkdir(parents=True, exist_ok=True)

GIB = 2**30
PHASES = ["draft_ms", "verification_ms", "commit_ms", "update_ms"]
KEY = ["method", "dataset", "k", "rank", "sample_idx"]
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

manifest = json.loads((run_dir / "manifest.json").read_text())
workers = pd.DataFrame([json.loads(p.read_text()) for p in sorted(run_dir.glob("workers/*.json"))])
samples = pd.DataFrame(read_jsonl(sorted(run_dir.glob("samples.*.jsonl"))))
steps = pd.DataFrame(read_jsonl(sorted(run_dir.glob("steps.*.jsonl"))))
summary_path = run_dir / "summary.jsonl"
summary = pd.DataFrame(read_jsonl([summary_path])) if summary_path.is_file() else pd.DataFrame()
if samples.empty or steps.empty:
    raise RuntimeError(f"{run_dir} has no sample/step records yet")

config = manifest["config"]
METHODS = [m["name"] for m in config["methods"] if m["name"] in set(samples["method"])]
METHODS += sorted(set(samples["method"]) - set(METHODS))
SPEC_METHODS = [m for m in METHODS if m != "ar"]
HAS_AR = "ar" in METHODS
DATASETS = [d for d, _ in config["datasets"] if d in set(samples["dataset"])]
DATASETS += sorted(set(samples["dataset"]) - set(DATASETS))
KS = sorted(int(k) for k in samples.loc[samples["method"] != "ar", "k"].unique())

gen = config["generation"]
display(Markdown(
    f"**Run** `{manifest['run_id']}` · git `{manifest['git']['commit'][:10]}`"
    f"{' (dirty)' if manifest['git']['dirty'] else ''} · GPU: {', '.join(sorted(set(workers['gpu_name'])))}"
    f" × {int(workers['world_size'].max())}  \n"
    f"**Target** `{config['target_name_or_path']}` · temperature {gen['temperature']} · "
    f"max_new_tokens {gen['max_new_tokens']} · seed {gen['seed']} · warm-up {config['warmup_samples']} "
    f"samples · step timing {'on' if config['step_timing'] else 'off'}  \n"
    f"**Methods** {', '.join(METHODS)} · **datasets** {', '.join(DATASETS)} · **K** {KS}  \n"
    f"**Overrides** {manifest.get('opts') or 'none'} · torch {manifest['versions']['torch']}, "
    f"transformers {manifest['versions']['transformers']}"
))
counts = samples.groupby(["method", "dataset", "k"]).size().unstack("dataset").reindex(columns=DATASETS)
display(counts.rename_axis(columns="samples per dataset"))

## 1. Data integrity

Every check below is computed from the stored records. A failed check does not stop the notebook,
but the affected numbers should not be reported before the cause is understood.

In [ ]:
checks = []


def check(name, ok, detail=""):
    detail = detail if len(detail) <= 160 else detail[:157] + "..."
    checks.append({"check": name, "ok": "✓" if ok else "✗ FAIL", "detail": detail})


# Per-sample sums of the step records.
step_sums = steps.groupby(KEY).agg(
    steps_counted=("step", "size"),
    drafted=("drafted_len", "sum"),
    effective=("effective_len", "sum"),
    accepted=("accepted", "sum"),
    committed=("committed", "sum"),
    **{f"{p}_steps": (p, "sum") for p in PHASES},
)
s = samples.merge(step_sums, left_on=KEY, right_index=True, how="left")
for col in ["steps_counted", "drafted", "effective", "accepted", "committed"]:
    s[col] = s[col].fillna(0).astype(int)

check("step count matches num_steps", (s["steps_counted"] == s["num_steps"]).all())
inv = steps.eval("0 <= accepted <= effective_len <= drafted_len") & steps["committed"].sub(steps["accepted"]).isin([0, 1])
check("0 ≤ accepted ≤ effective_len ≤ drafted_len; committed − accepted ∈ {0, 1}", inv.all(),
      f"{(~inv).sum()} violating steps")
done = s["stop_reason"].eq("stop_token")
tok_ok = np.where(done, 1 + s["committed"] == s["generated_len"], 1 + s["committed"] >= s["generated_len"])
check("1 + Σ committed = generated_len (≥ when length-limited)", tok_ok.all())
timed = s[PHASES].notna().all(axis=1)
check("step timing present", timed.all(), f"{(~timed).sum()} samples without phase timing")
phase_ok = all(np.allclose(s.loc[timed, p], s.loc[timed, f"{p}_steps"], rtol=1e-6, atol=1e-6) for p in PHASES)
check("sample phase sums equal Σ step phases", phase_ok)
mismatches = check_parity(run_dir) if summary_path.is_file() else ["summary.jsonl missing"]
check("summary.jsonl reproduced from steps (experiments.metrics.check_parity)", not mismatches,
      f"{len(mismatches)} mismatches, first: {mismatches[0]}" if mismatches else "")
sha = samples.groupby(["dataset", "sample_idx"])["prompt_sha1"].nunique()
check("same prompt for a sample_idx across methods and K", (sha == 1).all(), f"{(sha > 1).sum()} differing prompts")
sets = samples.groupby(["dataset", "method", "k"])["sample_idx"].apply(frozenset)
same_sets = sets.groupby(level="dataset").nunique().eq(1)
check("identical sample sets for every (method, K) of a dataset", same_sets.all(),
      "differs for " + ", ".join(same_sets.index[~same_sets]) if not same_sets.all() else "")
gpu_ms = s["target_prefill_ms"] + s["init_context_ms"].fillna(0) + s[PHASES].sum(axis=1)
host_frac = (s["wall_ms"] - gpu_ms) / s["wall_ms"]
check("wall_ms ≥ GPU-stream timeline (prefill + init + decode)", (host_frac > -1e-3).all(),
      f"host-only share of wall time: median {host_frac.median():.2%}, max {host_frac.max():.2%}")
check("AR baseline present", HAS_AR, "" if HAS_AR else "speedup and memory-over-AR columns will be empty")
check("warm-up enabled", config["warmup_samples"] > 0, f"{config['warmup_samples']} samples per (method, K, dataset)")
check("clean git tree at run time", not manifest["git"]["dirty"], f"commit {manifest['git']['commit'][:10]}")
display(pd.DataFrame(checks).set_index("check"))

## 2. Metric definitions and aggregation

Field semantics (verified in `experiments/records.py`, `instrumentation.py`, `metrics.py`):

| Record field | Meaning |
|---|---|
| `effective_len` | draft tokens submitted for verification in a step (after stop-token truncation); `K` for full proposals, 0 for AR |
| `accepted` | draft tokens accepted by verification (excludes the bonus token) |
| `committed` | tokens appended this step = `accepted` + 1 bonus/corrected target token (just `accepted` if an accepted draft token is a stop token) |
| `generated_len` | output tokens, **including the first token sampled from the prefill logits** |
| `draft_ms`, `verification_ms`, `update_ms`, `commit_ms` | CUDA-event phases that tile each step; `commit_ms` is the remaining per-step overhead (token commit, KV crop, stop check, bookkeeping) |
| `target_prefill_ms`, `init_context_ms` | target prompt forward; drafter state initialization from the prompt. Both excluded from decode |
| `wall_ms` | host wall clock for the whole sample (= GPU timeline + host-only time) |
| `summary.metrics.acceptance_length` | Σ accepted / Σ steps — **accepted draft tokens per step, no bonus token** |
| `summary.deepspec_metrics.acceptance_length` | DeepSpec's convention: Σ committed / Σ steps (includes the bonus token) |

Derived quantities used in this report:

| Report metric | Definition |
|---|---|
| decode time | `draft + verification + commit + update` (per sample) |
| decode tokens | `generated_len − 1` — tokens produced by the decode loop and kept. On length-limited samples the last step can commit tokens beyond `max_new_tokens` that are trimmed; they are not counted |
| effective draft length | Σ effective_len / Σ steps |
| accepted length | Σ accepted / Σ steps (equals `summary.metrics.acceptance_length`; asserted below) |
| acceptance rate | Σ accepted / Σ effective_len |
| committed / step | Σ committed / Σ steps (= DeepSpec's "acceptance length") |
| kept tokens / step | Σ decode tokens / Σ steps (committed / step minus tokens trimmed at `max_new_tokens`; ≈ 1 for AR) |
| decode latency | Σ decode time / Σ decode tokens, ms per generated token |
| throughput | 1000 / decode latency, tokens/s (single sequence, bsz 1) |
| speedup | AR decode latency / method decode latency on the same dataset (same prompts) |
| incremental decode memory | `decode_peak − prefill_peak` per sample |
| additional VRAM | method decode peak − AR decode peak, paired by sample |

**Aggregation.** Each `(method, dataset, K)` cell is aggregated separately. Ratio metrics are
*ratios of sums* over all steps/tokens of the cell (pooled), which is what the summary metrics and
throughput mean; memory metrics are means of per-sample peaks. Datasets are never pooled: the
"all datasets" rows are an equal-weight (macro) mean of the per-dataset values (a geometric mean
for speedup), so GSM8K/Math500 with 500 samples do not drown out AIME25 with 30.

**Uncertainty.** 95% percentile-bootstrap intervals, resampling *samples* (steps within one sample
are correlated, so steps are not resampled independently). Every cell of a dataset uses the same
resampled sample indices, so comparisons with AR (speedup, additional VRAM) are paired by prompt.
Macro rows average the per-dataset bootstrap replicates. The intervals reflect prompt-to-prompt
variability and sampling randomness at temperature 1.0; they do not cover run-to-run GPU timing
drift (≈ ±3% per README).

In [ ]:
s["decode_ms"] = s[PHASES].sum(axis=1, min_count=len(PHASES))
s["decode_tokens"] = (s["generated_len"] - 1).clip(lower=0)
s["has_decode"] = s["init_context_ms"].notna().astype(float)
s["e2e_ms"] = s["target_prefill_ms"] + s["init_context_ms"].fillna(0) + s["decode_ms"]
s["host_ms"] = s["wall_ms"] - s["e2e_ms"]
s["context_len"] = s["prompt_len"] + s["generated_len"]
s["rejected"] = s["effective"] - s["accepted"]
s["prefill_peak_gib"] = s["prefill_peak_bytes"] / GIB
s["decode_peak_gib"] = s["decode_peak_bytes"] / GIB
s["incr_decode_gib"] = s["decode_peak_gib"] - s["prefill_peak_gib"]
# Memory above what was allocated after loading the models (≈ weights): KV caches, activations, buffers.
post_load = workers.set_index(["method", "rank"])["post_load_allocated_bytes"]
s["post_load_bytes"] = [post_load.get((m, r), np.nan) for m, r in zip(s["method"], s["rank"])]
s["decode_dyn_mib"] = (s["decode_peak_bytes"] - s["post_load_bytes"]) / 2**20
s["prefill_dyn_mib"] = (s["prefill_peak_bytes"] - s["post_load_bytes"]) / 2**20
s["length_limited"] = s["stop_reason"].eq("length").astype(float)
s["one"] = 1.0

# Step 0 of DFlash/DSpark also projects the prompt context inside `propose`: a one-off startup cost
# that the step records count as decode. Measured as step-0 draft time minus the median later step.
first = steps[steps["step"] == 0].set_index(KEY)["draft_ms"]
later = steps[steps["step"] > 0].groupby(KEY)["draft_ms"].median()
s = s.merge((first - later).rename("step0_draft_excess_ms"), left_on=KEY, right_index=True, how="left")

# name: (numerator, denominator); every metric is Σ numerator / Σ denominator over samples.
RATIO_METRICS = {
    "eff_draft_len": ("effective", "num_steps"),
    "accepted_len": ("accepted", "num_steps"),
    "acceptance_rate": ("accepted", "effective"),
    "committed_per_step": ("committed", "num_steps"),
    "kept_per_step": ("decode_tokens", "num_steps"),
    "ms_per_token": ("decode_ms", "decode_tokens"),
    "ms_per_step": ("decode_ms", "num_steps"),
    **{f"{p}_per_token": (p, "decode_tokens") for p in PHASES},
    **{f"{p}_per_step": (p, "num_steps") for p in PHASES},
    "e2e_ms_per_token": ("e2e_ms", "generated_len"),
    "host_ms_per_token": ("host_ms", "generated_len"),
    "target_fwd_per_token": ("num_steps", "decode_tokens"),
    "draft_tokens_per_token": ("effective", "decode_tokens"),
    "rejected_per_token": ("rejected", "decode_tokens"),
    # Per-sample means.
    "prefill_ms": ("target_prefill_ms", "one"),
    "init_context_ms": ("init_context_ms", "has_decode"),
    "step0_draft_excess_ms": ("step0_draft_excess_ms", "has_decode"),
    "prefill_peak_gib": ("prefill_peak_gib", "one"),
    "decode_peak_gib": ("decode_peak_gib", "has_decode"),
    "incr_decode_gib": ("incr_decode_gib", "has_decode"),
    "prompt_len": ("prompt_len", "one"),
    "generated_len": ("generated_len", "one"),
    "length_limited_share": ("length_limited", "one"),
}

alpha = 1 - CI_LEVEL
records, reps_store = [], {}


def put(key, name, point, reps):
    reps_store[(*key, name)] = reps
    lo, hi = (np.nanpercentile(reps, [100 * alpha / 2, 100 * (1 - alpha / 2)])
              if np.isfinite(reps).any() else (np.nan, np.nan))
    records.append({"method": key[0], "dataset": key[1], "k": key[2], "metric": name,
                    "value": point, "lo": lo, "hi": hi})


with np.errstate(divide="ignore", invalid="ignore"), warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    for dataset in DATASETS:
        ds = s[s["dataset"] == dataset]
        universe = np.sort(ds["sample_idx"].unique())
        rng = np.random.default_rng([SEED, zlib.crc32(dataset.encode())])
        idx = rng.integers(0, len(universe), size=(N_BOOT, len(universe)))
        for (method, k), cell in ds.groupby(["method", "k"]):
            cell = cell.set_index("sample_idx").reindex(universe)  # missing samples contribute 0 / 0
            key = (method, dataset, int(k))
            for name, (num_col, den_col) in RATIO_METRICS.items():
                num, den = cell[num_col].to_numpy(float), cell[den_col].to_numpy(float)
                valid = np.isfinite(num) & np.isfinite(den)
                num, den = np.where(valid, num, 0.0), np.where(valid, den, 0.0)
                point = num.sum() / den.sum() if den.sum() > 0 else np.nan
                put(key, name, point, num[idx].sum(1) / den[idx].sum(1))
            put(key, "decode_peak_gib_max", cell["decode_peak_gib"].max(), np.full(N_BOOT, np.nan))

    def value(key, name):
        row = next(r for r in reversed(records) if (r["method"], r["dataset"], r["k"], r["metric"]) == (*key, name))
        return row["value"]

    # Derived metrics: throughput, speedup and memory over AR (paired through the shared indices).
    for key in list({(r["method"], r["dataset"], r["k"]) for r in records}):
        method, dataset, k = key
        put(key, "throughput", 1000 / value(key, "ms_per_token"), 1000 / reps_store[(*key, "ms_per_token")])
        put(key, "e2e_throughput", 1000 / value(key, "e2e_ms_per_token"), 1000 / reps_store[(*key, "e2e_ms_per_token")])
        ar = ("ar", dataset, 0)
        if HAS_AR and (*ar, "ms_per_token") in reps_store:
            for name, base in [("speedup", "ms_per_token"), ("e2e_speedup", "e2e_ms_per_token")]:
                put(key, name, value(ar, base) / value(key, base), reps_store[(*ar, base)] / reps_store[(*key, base)])
            put(key, "step_cost_ratio", value(key, "ms_per_step") / value(ar, "ms_per_step"),
                reps_store[(*key, "ms_per_step")] / reps_store[(*ar, "ms_per_step")])
            put(key, "extra_vram_gib", value(key, "decode_peak_gib") - value(ar, "decode_peak_gib"),
                reps_store[(*key, "decode_peak_gib")] - reps_store[(*ar, "decode_peak_gib")])

    # Macro rows over datasets (only for (method, K) present on every dataset).
    GEOMETRIC = {"speedup", "e2e_speedup", "step_cost_ratio"}
    long = pd.DataFrame(records)
    for (method, k, name), grp in long.groupby(["method", "k", "metric"]):
        if set(grp["dataset"]) != set(DATASETS) or name == "decode_peak_gib_max":
            continue
        point = grp["value"].to_numpy()
        reps = np.stack([reps_store[(method, d, k, name)] for d in grp["dataset"]])
        if name in GEOMETRIC:
            put((method, "all (macro)", k), name, np.exp(np.log(point).mean()), np.exp(np.log(reps).mean(0)))
        else:
            put((method, "all (macro)", k), name, point.mean(), reps.mean(0))

long = pd.DataFrame(records)
stats = long.pivot_table(index=["method", "dataset", "k"], columns="metric", values=["value", "lo", "hi"], dropna=False)
DATASETS_ALL = DATASETS + (["all (macro)"] if len(DATASETS) > 1 else [])


def get(method, dataset, k, name, field="value"):
    try:
        return stats.loc[(method, dataset, k), (field, name)]
    except KeyError:
        return np.nan


# Cross-check against summary.jsonl: accepted length and DeepSpec's committed/step are the same numbers.
if not summary.empty and "deepspec_metrics" in summary:
    differing = [
        key for key in ((r.method, r.dataset, int(r.k), r) for r in summary.itertuples())
        if not (np.isclose(get(*key[:3], "accepted_len"), key[3].metrics["acceptance_length"])
                and np.isclose(get(*key[:3], "committed_per_step"), key[3].deepspec_metrics["acceptance_length"]))
    ]
    if differing:
        warnings.warn(f"{len(differing)} summary.jsonl cells differ from the records, e.g. {differing[0][:3]}")
    else:
        print("accepted_len and committed_per_step match summary.jsonl for all", len(summary), "cells")


def save_table(df, name):
    if save_dir is not None:
        df.to_csv(save_dir / f"{name}.csv")


save_table(long.set_index(["method", "dataset", "k", "metric"]), "all_metrics_long")

In [ ]:
# ---- Plot style ---------------------------------------------------------------------------------
# Categorical slots 1-3 of the reference palette (validated all-pairs, so safe on scatter plots);
# AR is a neutral reference, not a series. Markers give a second, non-color encoding.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
MARKERS = ["o", "s", "^", "D", "v", "P", "X", "*"]
LABELS = {"ar": "AR baseline", "eagle3": "EAGLE-3", "dflash": "DFlash", "dspark": "DSpark"}
AR_COLOR = "#52514e"
STYLE = {m: dict(color=SERIES[i % len(SERIES)], marker=MARKERS[i % len(MARKERS)], label=LABELS.get(m, m))
         for i, m in enumerate(SPEC_METHODS)}
STYLE["ar"] = dict(color=AR_COLOR, marker="o", label=LABELS["ar"])
# Latency phases (validated as an adjacent stack).
PHASE_COLORS = {"verification_ms": "#6250d6", "draft_ms": "#eda100", "update_ms": "#e87ba4", "commit_ms": "#008300"}
PHASE_LABELS = {"verification_ms": "verification", "draft_ms": "draft", "update_ms": "drafter update",
                "commit_ms": "commit / other"}

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "savefig.facecolor": "#fcfcfb",
    "font.size": 9.5, "axes.titlesize": 10.5, "axes.titleweight": "bold", "axes.labelcolor": "#52514e",
    "axes.edgecolor": "#c3c2b7", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.axisbelow": True,
    "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False, "lines.linewidth": 2,
    "lines.markersize": 7,
})


def save_fig(fig, name):
    if save_dir is not None:
        fig.savefig(save_dir / f"{name}.png", bbox_inches="tight", dpi=150)


def panels(n, *, height=3.3, width=4.3, sharey=False):
    fig, axes = plt.subplots(1, n, figsize=(width * n, height), sharey=sharey, squeeze=False, constrained_layout=True)
    return fig, axes[0]


def ar_reference(ax, dataset, name, text="AR"):
    if not HAS_AR:
        return
    y, lo, hi = (get("ar", dataset, 0, name, f) for f in ("value", "lo", "hi"))
    if np.isfinite(y):
        ax.axhline(y, color=AR_COLOR, ls="--", lw=1.5, zorder=1)
        if np.isfinite(lo):
            ax.axhspan(lo, hi, color=AR_COLOR, alpha=0.10, lw=0, zorder=0)
        ax.annotate(f"{text} {y:.3g}", (0.01, y), xycoords=("axes fraction", "data"), xytext=(2, 3),
                    textcoords="offset points", color=AR_COLOR, fontsize=8.5)


def metric_vs_k(name, ylabel, title, *, ar_ref=False, ref=None, datasets=None, zero=True, fname=None):
    """One panel per dataset; K on x, one line per speculative method with bootstrap CI error bars."""
    datasets = datasets or DATASETS_ALL
    fig, axes = panels(len(datasets))
    for ax, dataset in zip(axes, datasets):
        if ref is not None:
            ref(ax)
        if ar_ref:
            ar_reference(ax, dataset, name)
        for m in SPEC_METHODS:
            y, lo, hi = (np.array([get(m, dataset, k, name, f) for k in KS], float) for f in ("value", "lo", "hi"))
            if not np.isfinite(y).any():
                continue
            err = np.vstack([y - lo, hi - y])
            ax.errorbar(KS, y, yerr=np.where(np.isfinite(err), err, 0), capsize=3, elinewidth=1.2, **STYLE[m])
        ax.set_title(dataset)
        ax.set_xlabel("draft length K")
        ax.set_xticks(KS)
        ax.set_xlim(min(KS) - 0.6, max(KS) + 0.6)
        if zero:
            ax.set_ylim(bottom=0)
    axes[0].set_ylabel(ylabel)
    handles, labels = axes[0].get_legend_handles_labels()
    if not labels:
        plt.close(fig)
        print(f"{title}: no data in this run (skipped)")
        return
    fig.legend(handles, labels, loc="outside upper right", ncol=len(labels))
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    save_fig(fig, fname or name)
    plt.show()


def fmt(name, digits=2, ci=False):
    def f(row):
        v = row.get(("value", name), np.nan)
        if not np.isfinite(v):
            return "—"
        text = f"{v:.{digits}f}"
        lo, hi = row.get(("lo", name), np.nan), row.get(("hi", name), np.nan)
        if ci and np.isfinite(lo):
            text += f" [{lo:.{digits}f}, {hi:.{digits}f}]"
        return text
    return f


def table(columns, *, methods=None, datasets=None):
    """columns: {header: (metric, digits, show_ci)}; one row per (method, dataset, K)."""
    methods = methods or METHODS
    datasets = datasets or DATASETS_ALL
    order = [(m, d, k) for d in datasets for m in methods for k in ([0] if m == "ar" else KS)
             if (m, d, k) in stats.index]
    sub = stats.loc[order]
    out = pd.DataFrame({h: sub.apply(fmt(*spec), axis=1) for h, spec in columns.items()}, index=sub.index)
    out.index = pd.MultiIndex.from_tuples([(d, LABELS.get(m, m), "—" if m == "ar" else k) for m, d, k in order],
                                          names=["dataset", "method", "K"])
    return out

## 3. Performance comparison

### 3.1 Main performance table

Decode-only numbers (prefill and drafter initialization excluded). Brackets are 95% bootstrap CIs.
`committed / step` is the decoding progress per target forward (DeepSpec's "acceptance length");
`accepted length` excludes the bonus token. `length-limited` is the share of samples that hit
`max_new_tokens` — high values mean outputs are truncated and context lengths are capped.

In [ ]:
main_table = table({
    "eff. draft len": ("eff_draft_len", 2, False),
    "accepted len": ("accepted_len", 2, True),
    "accept. rate": ("acceptance_rate", 3, False),
    "committed / step": ("committed_per_step", 2, False),
    "ms / token": ("ms_per_token", 2, True),
    "tokens / s": ("throughput", 1, False),
    "speedup vs AR": ("speedup", 2, True),
    "gen. len": ("generated_len", 0, False),
    "length-limited": ("length_limited_share", 2, False),
})
save_table(main_table, "main_performance")
display(main_table)

### 3.2 Acceptance vs draft length

Mean accepted draft tokens per speculative step. The dotted diagonal `y = K` is the ceiling (every
draft token accepted). Acceptance rate normalizes by the draft tokens actually submitted
(`effective_len`, which can be below `K` when EAGLE-3's chain stops at a stop token).

In [ ]:
def diagonal(ax):
    ks = np.array([min(KS) - 0.6, max(KS) + 0.6])
    ax.plot(ks, ks, ls=":", color="#a3a29b", lw=1.2, zorder=1)
    ax.annotate("y = K (all accepted)", (max(KS), max(KS)), xytext=(-6, 4), textcoords="offset points",
                ha="right", color="#8a8983", fontsize=8)


metric_vs_k("accepted_len", "accepted draft tokens / step", "Accepted length vs draft length", ref=diagonal)
metric_vs_k("acceptance_rate", "accepted / effective draft tokens", "Acceptance rate vs draft length")

Per-position acceptance: share of steps whose first `i` draft tokens were all accepted, among steps
that drafted at least `i` tokens (DeepSpec's `accept_rates_by_position`). It shows where along the
draft the methods lose tokens, and how running DFlash/DSpark at `K < 7` (off their training block
size) changes acceptance at the same positions.

In [ ]:
def position_rates(sub):
    max_pos = int(sub["effective_len"].max()) if len(sub) else 0
    pos = np.arange(1, max_pos + 1)
    eff, acc = sub["effective_len"].to_numpy()[:, None], sub["accepted"].to_numpy()[:, None]
    return pos, (acc >= pos).sum(0) / np.maximum((eff >= pos).sum(0), 1)


fig, axes = plt.subplots(len(DATASETS), len(KS), figsize=(3.6 * len(KS), 2.7 * len(DATASETS)),
                         squeeze=False, sharey=True, constrained_layout=True)
spec_steps = steps[steps["method"] != "ar"]
for i, dataset in enumerate(DATASETS):
    for j, k in enumerate(KS):
        ax = axes[i, j]
        for m in SPEC_METHODS:
            sub = spec_steps[(spec_steps["method"] == m) & (spec_steps["dataset"] == dataset) & (spec_steps["k"] == k)]
            if len(sub):
                pos, rate = position_rates(sub)
                ax.plot(pos, rate, **STYLE[m])
        ax.set_ylim(0, 1.02)
        ax.set_xticks(range(1, k + 1))
        ax.set_title(f"{dataset} · K={k}", fontsize=9.5)
        if j == 0:
            ax.set_ylabel("P(first i accepted)")
        if i == len(DATASETS) - 1:
            ax.set_xlabel("draft position i")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper right", ncol=len(labels))
save_fig(fig, "acceptance_by_position")
plt.show()

### 3.3 Decode latency and speedup vs draft length

Decode latency per generated token; the dashed line and band are the AR baseline and its CI on the
same prompts. Note that at temperature 1.0 methods generate different continuations, so their
context lengths differ slightly; per-token attention cost grows with context, which this
normalization does not remove (compare `gen. len` in the main table).

In [ ]:
metric_vs_k("ms_per_token", "decode ms / generated token", "Decode latency vs draft length", ar_ref=True)
metric_vs_k("speedup", "speedup over AR (decode)", "Decode speedup vs draft length", zero=False,
            ref=lambda ax: ax.axhline(1, color=AR_COLOR, ls="--", lw=1.2))

### 3.4 Latency breakdown per generated token

The four step phases tile the decode timeline, so they sum exactly to the decode latency.
`commit / other` is everything in a step outside propose / verify / update: token commit, KV-cache
crop, stop-token check and loop bookkeeping.

In [ ]:
breakdown = table({
    "draft": ("draft_ms_per_token", 2, False),
    "verification": ("verification_ms_per_token", 2, False),
    "drafter update": ("update_ms_per_token", 2, False),
    "commit / other": ("commit_ms_per_token", 2, False),
    "decode total": ("ms_per_token", 2, True),
})
save_table(breakdown, "latency_breakdown_ms_per_token")
display(breakdown)

stack = ["verification_ms", "draft_ms", "update_ms", "commit_ms"]
configs = ([("ar", 0)] if HAS_AR else []) + [(m, k) for m in SPEC_METHODS for k in KS]
fig, axes = plt.subplots(len(DATASETS_ALL), 1, figsize=(max(7, 0.75 * len(configs) + 2), 2.9 * len(DATASETS_ALL)),
                         squeeze=False, constrained_layout=True)
for ax, dataset in zip(axes[:, 0], DATASETS_ALL):
    x = np.arange(len(configs))
    bottom = np.zeros(len(configs))
    for phase in stack:
        h = np.array([get(m, dataset, k, f"{phase}_per_token") for m, k in configs], float)
        h = np.nan_to_num(h)
        ax.bar(x, h, bottom=bottom, width=0.7, color=PHASE_COLORS[phase], edgecolor="#fcfcfb", linewidth=1,
               label=PHASE_LABELS[phase])
        bottom += h
    for xi, total in zip(x, bottom):
        if total > 0:
            ax.annotate(f"{total:.1f}", (xi, total), xytext=(0, 2), textcoords="offset points", ha="center",
                        fontsize=8, color="#52514e")
    ax.set_xticks(x, ["AR" if m == "ar" else f"{LABELS.get(m, m)}\nK={k}" for m, k in configs], fontsize=8)
    ax.set_ylabel("ms / generated token")
    ax.set_title(dataset, loc="left")
    ax.grid(axis="x", visible=False)
    ax.set_ylim(0, bottom.max() * 1.15 if bottom.max() > 0 else 1)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper right", ncol=4)
fig.suptitle("Decode latency breakdown", x=0.01, ha="left", fontweight="bold")
save_fig(fig, "latency_breakdown")
plt.show()

### 3.5 Why a configuration is faster or slower

Decode latency per token factorizes exactly (ratios of sums):

$$\text{ms/token} = \frac{\text{ms/step}}{\text{kept tokens/step}}, \qquad
\text{speedup} = \frac{\text{kept tokens/step}}{\text{step cost ratio}}, \quad
\text{step cost ratio} = \frac{\text{ms/step}_{\text{method}}}{\text{ms/step}_{\text{AR}}}$$

A larger `K` raises tokens per step (numerator) but also the draft time and the width of the
verification forward (denominator). The best `K` is where the extra accepted tokens still outpay
the extra step cost. `kept tokens / step` differs from `committed / step` only by tokens trimmed at
`max_new_tokens`: DeepSpec's loop runs one step past the limit on length-limited samples and trims,
so even AR is slightly below 1 token/step (one discarded forward per length-limited sample, ≈ 0.8%
at 128 tokens and < 0.05% at 2048). Every method pays this final step; its tokens are not counted.

In [ ]:
explain = table({
    "committed / step": ("committed_per_step", 2, False),
    "kept tokens / step": ("kept_per_step", 2, False),
    "ms / step": ("ms_per_step", 1, False),
    "draft ms / step": ("draft_ms_per_step", 1, False),
    "verify ms / step": ("verification_ms_per_step", 1, False),
    "step cost ratio": ("step_cost_ratio", 2, False),
    "speedup": ("speedup", 2, False),
    "ms / token": ("ms_per_token", 2, False),
})
save_table(explain, "step_level_explanation")
display(explain)

fig, axes = plt.subplots(2, len(DATASETS_ALL), figsize=(4.3 * len(DATASETS_ALL), 6.2), squeeze=False,
                         constrained_layout=True)
for j, dataset in enumerate(DATASETS_ALL):
    for i, (name, ylabel) in enumerate([("kept_per_step", "tokens / step"), ("ms_per_step", "ms / speculative step")]):
        ax = axes[i, j]
        ar_reference(ax, dataset, name)
        for m in SPEC_METHODS:
            y, lo, hi = (np.array([get(m, dataset, k, name, f) for k in KS], float) for f in ("value", "lo", "hi"))
            ax.errorbar(KS, y, yerr=np.nan_to_num(np.vstack([y - lo, hi - y])), capsize=3, elinewidth=1.2, **STYLE[m])
        ax.set_xticks(KS)
        ax.set_xlim(min(KS) - 0.6, max(KS) + 0.6)
        ax.set_ylim(bottom=0)
        if i == 0:
            ax.set_title(dataset)
        else:
            ax.set_xlabel("draft length K")
        if j == 0:
            ax.set_ylabel(ylabel)
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper right", ncol=len(labels))
fig.suptitle("Tokens per step (gain) and step latency (cost)", x=0.01, ha="left", fontweight="bold")
save_fig(fig, "step_gain_vs_cost")
plt.show()

## 4. Startup costs (excluded from decode latency)

Per-sample means. `drafter init` is `init_context_ms`. `step-0 draft excess` is the extra draft time
of the first step over the median later step: for DFlash/DSpark it contains projecting the prompt
context, so it is a startup cost that the step records still count as decode (it is small relative
to the decode time of a sample, but it is shown here so its size is known). End-to-end numbers
include prefill, drafter init and decode, normalized by all generated tokens; host-only time is
`wall_ms` minus the GPU-stream timeline.

In [ ]:
startup = table({
    "prompt len": ("prompt_len", 0, False),
    "target prefill ms": ("prefill_ms", 1, False),
    "drafter init ms": ("init_context_ms", 2, False),
    "step-0 draft excess ms": ("step0_draft_excess_ms", 2, False),
    "host-only ms / token": ("host_ms_per_token", 3, False),
    "e2e ms / token": ("e2e_ms_per_token", 2, False),
    "e2e tokens / s": ("e2e_throughput", 1, False),
    "e2e speedup": ("e2e_speedup", 2, True),
})
save_table(startup, "startup_costs")
display(startup)

## 5. Resource efficiency

Memory is PyTorch's peak *allocated* memory (`max_memory_allocated`), not reserved/driver memory.
`prefill peak` covers target prefill + drafter init; `decode peak` covers decode only (statistics
are reset after prefill), so it includes all weights plus the KV caches at their largest.

### 5.1 Models

In [ ]:
models = workers.sort_values("rank").drop_duplicates("method").set_index("method").reindex(METHODS)
ar_load = models.loc["ar", "post_load_allocated_bytes"] if HAS_AR else np.nan
model_table = pd.DataFrame({
    "drafter": models["draft_name_or_path"].fillna("—"),
    "drafter params (M)": (models["draft_params"] / 1e6).round(1),
    "drafter / target params": (models["draft_params"] / models["target_params"]).round(3),
    "allocated after load (GiB)": (models["post_load_allocated_bytes"] / GIB).round(2),
    "over AR after load (GiB)": ((models["post_load_allocated_bytes"] - ar_load) / GIB).round(2),
    "trained draft length": models["trained_draft_length"],
}).rename(index=lambda m: LABELS.get(m, m))
save_table(model_table, "models")
display(model_table)

### 5.2 Memory table

Means of per-sample peaks (GiB) with the configuration's maximum over samples. `additional VRAM`
is paired by prompt against AR's decode peak. Latency/throughput repeated for reference.

In [ ]:
memory_table = table({
    "prefill peak": ("prefill_peak_gib", 3, False),
    "decode peak": ("decode_peak_gib", 3, True),
    "decode peak (max)": ("decode_peak_gib_max", 3, False),
    "incremental decode": ("incr_decode_gib", 3, True),
    "additional VRAM vs AR": ("extra_vram_gib", 3, True),
    "ms / token": ("ms_per_token", 2, False),
    "tokens / s": ("throughput", 1, False),
    "speedup": ("speedup", 2, False),
})
save_table(memory_table, "memory")
display(memory_table)

In [ ]:
metric_vs_k("decode_peak_gib", "decode peak VRAM (GiB)", "Peak decode VRAM vs draft length", ar_ref=True, zero=False)
metric_vs_k("incr_decode_gib", "decode peak − prefill peak (GiB)", "Incremental decode memory vs draft length",
            ar_ref=True)

### 5.3 VRAM scaling with sequence length

Memory depends on sequence length, not on the dataset, so samples from all datasets are pooled here
(datasets only shift the length distribution). The y-axis is peak memory **above the allocation
right after model loading** (≈ weights), in MiB, i.e. KV caches, activations and work buffers; the
weights themselves are a constant offset (table 5.1) that would hide the scaling. Top: decode peak
vs final context length (`prompt_len + generated_len`), one panel per K, AR in every panel.
Bottom: prefill peak vs prompt length. The table gives least-squares slopes in MiB per 1k tokens and
the intercept in MiB (fixed buffers at zero length). The range of lengths is limited by
`max_new_tokens`; slopes from a narrow range are imprecise.

In [ ]:
def fit(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3 or np.ptp(x[ok]) == 0:
        return np.nan, np.nan
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    return slope, intercept


cols = len(KS)
fig, axes = plt.subplots(2, cols, figsize=(4.3 * cols, 6.4), squeeze=False, constrained_layout=True)
slopes = []
for j, k in enumerate(KS):
    for i, (xcol, ycol, xlabel) in enumerate([("context_len", "decode_dyn_mib", "prompt + generated tokens"),
                                              ("prompt_len", "prefill_dyn_mib", "prompt tokens")]):
        ax = axes[i, j]
        for m in METHODS:
            sub = s[(s["method"] == m) & (s["k"] == (0 if m == "ar" else k))]
            if sub.empty:
                continue
            x, y = sub[xcol].to_numpy(float), sub[ycol].to_numpy(float)
            st = STYLE[m]
            ax.scatter(x, y, s=16, color=st["color"], marker=st["marker"], alpha=0.55, edgecolors="#fcfcfb",
                       linewidths=0.5, label=st["label"])
            slope, intercept = fit(x, y)
            if np.isfinite(slope):
                xs = np.array([np.nanmin(x), np.nanmax(x)])
                ax.plot(xs, intercept + slope * xs, color=st["color"], lw=1.5)
            if m != "ar" or j == 0:
                slopes.append({"memory": ycol.replace("_dyn_mib", " peak"), "method": LABELS.get(m, m),
                               "K": 0 if m == "ar" else k, "MiB / 1k tokens": slope * 1000,
                               "intercept (MiB)": intercept, "samples": len(sub)})
        ax.set_xlabel(xlabel)
        if i == 0:
            ax.set_title(f"K={k}")
        if j == 0:
            ax.set_ylabel("decode peak above weights (MiB)" if i == 0 else "prefill peak above weights (MiB)")
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper right", ncol=len(labels), markerscale=2)
fig.suptitle("VRAM vs sequence length (all datasets pooled)", x=0.01, ha="left", fontweight="bold")
save_fig(fig, "vram_vs_length")
plt.show()

order = {LABELS.get(m, m): i for i, m in enumerate(METHODS)}
slope_table = (pd.DataFrame(slopes)
               .sort_values(["memory", "K", "method"], key=lambda c: c.map(order) if c.name == "method" else c)
               .replace({"K": {0: "—"}}).set_index(["memory", "method", "K"]).round(2))
save_table(slope_table, "vram_slopes")
display(slope_table)

### 5.4 Latency–memory trade-offs

Each point is one `(method, K)` configuration on one dataset (labelled with its K). Down-left is
better in the first figure; up-left is better in the second; thin lines join a method's K values in
order (speedup bought per additional GiB of
peak decode memory over AR, paired by prompt).

In [ ]:
def tradeoff(xname, yname, xlabel, ylabel, title, fname, *, ar_point=True, hline=None):
    fig, axes = panels(len(DATASETS_ALL), height=3.6)
    for ax, dataset in zip(axes, DATASETS_ALL):
        if hline is not None:
            ax.axhline(hline, color=AR_COLOR, ls="--", lw=1.2)
        if ar_point and HAS_AR:
            x, y = get("ar", dataset, 0, xname), get("ar", dataset, 0, yname)
            ax.scatter([x], [y], s=60, color=AR_COLOR, marker="o", edgecolors="#fcfcfb", linewidths=2,
                       label=LABELS["ar"], zorder=3)
        for m in SPEC_METHODS:
            st = STYLE[m]
            xs = np.array([get(m, dataset, k, xname) for k in KS], float)
            ys = np.array([get(m, dataset, k, yname) for k in KS], float)
            lo = np.array([get(m, dataset, k, yname, "lo") for k in KS], float)
            hi = np.array([get(m, dataset, k, yname, "hi") for k in KS], float)
            ax.plot(xs, ys, color=st["color"], lw=1, alpha=0.6, zorder=2)  # joins K values in order
            ax.errorbar(xs, ys, yerr=np.nan_to_num(np.vstack([ys - lo, hi - ys])), fmt="none", ecolor=st["color"],
                        elinewidth=1, capsize=2, alpha=0.7)
            ax.scatter(xs, ys, s=60, color=st["color"], marker=st["marker"], edgecolors="#fcfcfb", linewidths=2,
                       label=st["label"], zorder=3)
            for k, x, y in zip(KS, xs, ys):
                if np.isfinite(x) and np.isfinite(y):
                    ax.annotate(f"K{k}", (x, y), xytext=(5, 3), textcoords="offset points", fontsize=7.5,
                                color="#52514e")
        ax.set_title(dataset)
        ax.set_xlabel(xlabel)
        ax.margins(x=0.15, y=0.12)
    axes[0].set_ylabel(ylabel)
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="outside upper right", ncol=len(labels))
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold")
    save_fig(fig, fname)
    plt.show()


tradeoff("decode_peak_gib", "ms_per_token", "mean decode peak VRAM (GiB)", "decode ms / generated token",
         "Decode latency vs peak VRAM", "latency_vs_vram")
if HAS_AR:
    tradeoff("extra_vram_gib", "speedup", "additional decode VRAM over AR (GiB)", "speedup over AR",
             "Speedup vs additional VRAM", "speedup_vs_extra_vram", ar_point=False, hline=1)

### 5.5 Work per generated token

Measured work needed to produce one output token: target verification forwards, draft tokens
proposed and rejected (wasted drafting/verification width), and GPU time spent in the drafter and
in verification. AR does exactly one target forward and no drafting per token. Drafter parameter
counts are in 5.1; FLOP counts are not reported because they cannot be measured reliably from
these logs (they depend on each drafter's architecture and the attention context).

In [ ]:
work = table({
    "target forwards / token": ("target_fwd_per_token", 3, False),
    "draft tokens / token": ("draft_tokens_per_token", 2, False),
    "rejected draft tokens / token": ("rejected_per_token", 2, False),
    "draft ms / token": ("draft_ms_per_token", 2, False),
    "verify ms / token": ("verification_ms_per_token", 2, False),
    "additional VRAM (GiB)": ("extra_vram_gib", 3, False),
    "speedup": ("speedup", 2, False),
})
save_table(work, "work_per_token")
display(work)

## 6. Methodology notes and design decisions

- **Accepted vs committed.** "Accepted length" in this report is accepted *draft* tokens per step
  (`summary.metrics.acceptance_length`). DeepSpec's own `acceptance_length` counts committed tokens
  (accepted + bonus) and is reported here as `committed / step`; the two differ by ≈ 1.
- **Per-token normalization uses kept tokens** (`generated_len − 1`). The first output token is
  produced by prefill and its time is in `target_prefill_ms`; tokens committed beyond
  `max_new_tokens` on the last step are trimmed and not counted. Counting Σ committed instead would
  credit speculative methods for discarded tokens on length-limited samples.
- **Primary decode latency** is the sum of the four step phases. It excludes `target_prefill_ms` and
  `init_context_ms`; the DFlash/DSpark first-step context projection stays inside decode (it is in
  `propose`), and its size is shown separately in section 4.
- **Pooled ratios, not means of per-sample ratios.** Per-token latency and acceptance are ratios of
  sums within each `(method, dataset, K)` cell, so they equal the summary metrics and throughput over
  the cell. Datasets are combined only as an equal-weight macro mean (geometric for speedup).
- **Speedup and additional VRAM are paired.** Every method sees the same prompts (`prompt_sha1`
  checked), and the bootstrap resamples the same prompt indices for all cells of a dataset.
- **Remaining overhead = `commit_ms`.** Because phases share CUDA-event boundaries, decode has no
  unattributed GPU time; host-only time (outside the GPU-stream timeline) is reported per token in
  section 4 rather than mixed into decode.
- **Additions beyond the requested plots:** speedup vs K, acceptance by draft position, the
  gain/cost factorization of speedup (3.5), startup-cost table, model/parameter table, speedup vs
  additional VRAM and work per generated token (from `EXPERIMENTS.md`).
- **Caveat — sampling.** At temperature 1.0 continuations differ between methods, so generated
  lengths (and therefore context lengths during decode) are not identical; check `gen. len` and
  `length-limited` in the main table before attributing small latency differences to the method.